<a href="https://colab.research.google.com/github/Caleb-John1603/BIS_LabCodeDump/blob/main/WEEK%205_Grey%20Wolf%20Optimisation_1BM24CS095.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np

def evaluate_fitness(alloc, tasks, vms, w1=0.6, w2=0.4):
    num_vms = len(vms)
    cpu_load, mem_load = np.zeros(num_vms), np.zeros(num_vms)
    total_cost, makespan = 0, 0

    for t_idx, v_idx in enumerate(alloc):
        cpu_load[v_idx] += tasks[t_idx]["cpu"]
        mem_load[v_idx] += tasks[t_idx]["mem"]

        exec_time = tasks[t_idx]["len"] / vms[v_idx]["cpu"]
        makespan = max(makespan, exec_time)
        total_cost += exec_time * vms[v_idx]["cost"]

    penalty = sum(100000 for v in range(num_vms) if cpu_load[v] > vms[v]["cpu"] or mem_load[v] > vms[v]["mem"])
    return (w1 * makespan) + (w2 * total_cost) + penalty

def gwo_allocation(tasks, vms, num_wolves=30, max_iter=100):
    num_tasks, num_vms = len(tasks), len(vms)

    wolves = np.random.randint(0, num_vms, size=(num_wolves, num_tasks))
    fitness = np.array([evaluate_fitness(w, tasks, vms) for w in wolves])

    sorted_idxs = np.argsort(fitness)
    alpha_pos = wolves[sorted_idxs[0]].copy()
    alpha_fit = fitness[sorted_idxs[0]]
    beta_pos = wolves[sorted_idxs[1]].copy()
    beta_fit = fitness[sorted_idxs[1]]
    delta_pos = wolves[sorted_idxs[2]].copy()
    delta_fit = fitness[sorted_idxs[2]]

    for t in range(max_iter):
        a = 2.0 - (2.0 * t / max_iter)

        for i in range(num_wolves):
            # Generate random parameter arrays for all tasks at once
            A1 = 2 * a * np.random.rand(num_tasks) - a
            C1 = 2 * np.random.rand(num_tasks)
            X1 = alpha_pos - A1 * np.abs(C1 * alpha_pos - wolves[i])

            A2 = 2 * a * np.random.rand(num_tasks) - a
            C2 = 2 * np.random.rand(num_tasks)
            X2 = beta_pos - A2 * np.abs(C2 * beta_pos - wolves[i])

            A3 = 2 * a * np.random.rand(num_tasks) - a
            C3 = 2 * np.random.rand(num_tasks)
            X3 = delta_pos - A3 * np.abs(C3 * delta_pos - wolves[i])

            # Vectorised average, rounding, and clipping
            new_position = np.round((X1 + X2 + X3) / 3.0).astype(int)
            wolves[i] = np.clip(new_position, 0, num_vms - 1)

            fit = evaluate_fitness(wolves[i], tasks, vms)
            fitness[i] = fit

            if fit < alpha_fit:
                delta_fit, delta_pos = beta_fit, beta_pos.copy()
                beta_fit, beta_pos = alpha_fit, alpha_pos.copy()
                alpha_fit, alpha_pos = fit, wolves[i].copy()
            elif fit < beta_fit:
                delta_fit, delta_pos = beta_fit, beta_pos.copy()
                beta_fit, beta_pos = fit, wolves[i].copy()
            elif fit < delta_fit:
                delta_fit, delta_pos = fit, wolves[i].copy()

    return alpha_pos, alpha_fit

sample_tasks = [{"cpu": 2, "mem": 4, "len": 500} for _ in range(20)]
sample_vms = [{"cpu": 25, "mem": 32, "cost": 0.15} for _ in range(5)]

best_mapping, min_score = gwo_allocation(sample_tasks, sample_vms, num_wolves=20, max_iter=50)

for vm_id in range(len(sample_vms)):
    assigned_tasks = [t_idx for t_idx, assigned_vm in enumerate(best_mapping) if assigned_vm == vm_id]
    print(f"VM {vm_id} handles Tasks: {assigned_tasks}")



VM 0 handles Tasks: [14, 17]
VM 1 handles Tasks: [0, 2, 5, 8, 11, 12, 13]
VM 2 handles Tasks: [10, 16]
VM 3 handles Tasks: [1, 6, 7, 9, 18]
VM 4 handles Tasks: [3, 4, 15, 19]
